In [1]:
import os

In [2]:
pwd

'd:\\ML Projects\\Chest Cancer Classification\\chest-cancer-classifier\\research'

In [3]:
os.chdir("../")

In [4]:
pwd

'd:\\ML Projects\\Chest Cancer Classification\\chest-cancer-classifier'

In [5]:
from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True)
class PredictionConfig:
    root_dir: Path
    trained_model_path: Path
    updated_base_model_path: Path
    params_image_size: list

In [6]:
from cnnClassifier.constants import *
from cnnClassifier.utils.common import read_yaml

In [7]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)


    def get_prediction_config(self) -> PredictionConfig:
        training = self.config.training
        prepare_base_model = self.config.prepare_base_model
        params = self.params
       
        prediction_config = PredictionConfig(
            root_dir=Path(training.root_dir),
            trained_model_path=Path(training.trained_model_path),
            updated_base_model_path=Path(prepare_base_model.updated_base_model_path),
            params_image_size=params.IMAGE_SIZE,
        )

        return prediction_config
    

In [8]:
import os
import tensorflow as tf
import json
import numpy as np

In [20]:
class Prediction:
    
    def __init__(self, config):
        self.config = config

    def predict(self, img_path):
     
        # Path to your image
        img_path = "artifacts/data_ingestion/Chest-CT-Scan-data/normal/3.png"

        # Load the image with same size as training
        img = tf.keras.utils.load_img(
            img_path,
            target_size= self.config.params_image_size[:-1]  # e.g. (224, 224)
        )

        # Convert to array
        img_array = tf.keras.utils.img_to_array(img)

        # Scale (same as rescale=1./255 in generator)
        img_array = img_array / 255.0

        # Add batch dimension: (1, height, width, channels)
        img_array = np.expand_dims(img_array, axis=0)

        # Load later
        model = tf.keras.models.load_model("artifacts/training/model.keras")
        with open("model_with_classes/class_indices.json") as f:
            class_indices = json.load(f)

        # Predict
        pred = model.predict(img_array)

        # For binary classification
        if pred.shape[1] == 1:  # sigmoid output
            predicted_class = (pred > 0.5).astype("int32")[0][0]
        else:  # softmax output
            predicted_class = np.argmax(pred, axis=1)[0]

        # Map back to class name
        class_labels = list(class_indices.keys())
        print("Predicted class:", class_labels[predicted_class])



In [21]:
img_path = ""

In [22]:
try:
    config = ConfigurationManager()
    prediction_config = config.get_prediction_config()
    prediction = Prediction(config=prediction_config)
    prediction.predict(img_path)
    
except Exception as e:
    raise e

[2025-08-12 15:40:37,520: INFO: common: yaml file: config\config.yaml loaded successfully]
[2025-08-12 15:40:37,526: INFO: common: yaml file: params.yaml loaded successfully]
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 689ms/step
Predicted class: normal
